# 🧠 DSA Lab 09: Doubly Linked Lists, Circular Linked Lists and Their Applications
### Student Notebook

**Data Structures & Algorithms Lab Manual | University of Okara | Fall 2026**

Prepared by **Muhammad Ramzan**<br>
MPhil Scholar (Artificial Intelligence), PUCIT, University of the Punjab, Lahore<br>
Lab Engineer, Department of Computer Science, University of Okara

🔗 [GitHub](https://github.com/RamzanPUCIT) · [LinkedIn](https://www.linkedin.com/in/ramzan-ai) · [ResearchGate](https://www.researchgate.net/profile/Muhammad-Ramzan-105)

---
**How to use:** run each code cell with **Shift + Enter**, from top to bottom. Read the matching section in the Lab 09 manual, change values and re-run to see what happens.

> 📝 **Your job:** run the examples, then complete every `# TODO` in the Lab Tasks part. Each task has a **test cell**: when it prints ✅ your solution is correct. Write pseudocode as comments before coding!

## 📖 Part A: Doubly Linked List

In a **doubly linked list (DLL)**, each node has **three parts**: `prev` (reference to the previous node), `data`, and `next`.

**A doubly linked list**

```text
         +------+----+------+     +------+----+------+     +------+----+------+
None <---| prev | 10 | next |<--->| prev | 20 | next |<--->| prev | 30 | next |---> None
         +------+----+------+     +------+----+------+     +------+----+------+
           head                                              tail
```

|  | Singly linked list | Doubly linked list |
|---|---|---|
| References per node | 1 (next) | 2 (prev, next) |
| Traverse backwards | Not possible | Yes |
| Delete last node | O(n) | O(1) |
| Delete a given node (with reference) | O(n), need previous node | O(1) |
| Memory | Less | More (one extra reference per node) |
| Code complexity | Simpler | More pointers to update |

### Sentinel Nodes: A Professional Trick

Handling empty lists and the head/tail special cases makes linked-list code messy. Professionals add two **dummy (sentinel) nodes** that never hold real data: `head` sentinel at the start and `tail` sentinel at the end. Every real node is always **between two nodes**, so insertion and deletion have **no special cases**.

**DLL with sentinels**

```text
[HEAD] <-> [10] <-> [20] <-> [30] <-> [TAIL]       (HEAD and TAIL hold no data)
Empty list:  [HEAD] <-> [TAIL]
```

In [ ]:
class DNode:
    def __init__(self, data=None):
        self.data = data
        self.prev = None
        self.next = None

class DoublyLinkedList:
    def __init__(self):
        self.head = DNode()            # sentinel
        self.tail = DNode()            # sentinel
        self.head.next = self.tail
        self.tail.prev = self.head
        self.size = 0

    def _insert_between(self, data, before, after):
        node = DNode(data)
        node.prev, node.next = before, after
        before.next = node
        after.prev = node
        self.size += 1
        return node

    def _unlink(self, node):           # O(1) delete of a known node
        node.prev.next = node.next
        node.next.prev = node.prev
        self.size -= 1
        return node.data

    def push_front(self, data):
        return self._insert_between(data, self.head, self.head.next)

    def push_back(self, data):
        return self._insert_between(data, self.tail.prev, self.tail)

    def pop_front(self):
        if self.size == 0:
            raise IndexError("empty list")
        return self._unlink(self.head.next)

    def pop_back(self):
        if self.size == 0:
            raise IndexError("empty list")
        return self._unlink(self.tail.prev)

    def forward(self):
        out, cur = [], self.head.next
        while cur is not self.tail:
            out.append(cur.data); cur = cur.next
        return out

    def backward(self):
        out, cur = [], self.tail.prev
        while cur is not self.head:
            out.append(cur.data); cur = cur.prev
        return out

d = DoublyLinkedList()
for x in [20, 30, 40]:
    d.push_back(x)
d.push_front(10)
print("forward :", d.forward())
print("backward:", d.backward())
print("pop_back ->", d.pop_back(), "| pop_front ->", d.pop_front(), "|", d.forward())

### Inserting in the Middle: Four Links

To insert node N between nodes A and B, **four** references change: `N.prev = A`, `N.next = B`, `A.next = N`, `B.prev = N`. Set the new node’s links first, then update the neighbours.

```text
Before:  [A] <-> [B]
1. N.prev = A        2. N.next = B
3. A.next = N        4. B.prev = N
After:   [A] <-> [N] <-> [B]
```

### Deleting a Node in O(1)

If we hold a reference to a node (for example from a dictionary), we can remove it instantly: its previous node skips forward over it and its next node skips backward over it. This is impossible in O(1) with a singly linked list, and it is the key to the LRU cache.

In [ ]:
d2 = DoublyLinkedList()
handles = {name: d2.push_back(name) for name in ["Ali", "Sara", "Hamza", "Zoya"]}
d2._unlink(handles["Hamza"])           # O(1): no searching
print(d2.forward())

## ⭕ Part B: Circular Linked List

In a **circular linked list**, the last node points back to the **first** node instead of None. There is no end, so you can keep going round. It is convenient to keep a reference to the **tail**: then `tail.next` is the head, giving O(1) access to both ends.

**Circular singly linked list**

```text
      +----+     +----+     +----+     +----+
+---->| A  |---->| B  |---->| C  |---->| D  |----+
|     +----+     +----+     +----+     +----+    |
+------------------------------------------------+
                                         tail (tail.next is A)
```

> **⚠️ Careful with traversal**
>
> `while cur is not None` loops **forever** on a circular list. Instead, stop when you come back to the starting node.

In [ ]:
class CNode:
    def __init__(self, data):
        self.data = data
        self.next = None

class CircularList:
    def __init__(self):
        self.tail = None
        self.size = 0

    def append(self, data):
        node = CNode(data)
        if self.tail is None:
            node.next = node               # single node points to itself
        else:
            node.next = self.tail.next     # new node -> head
            self.tail.next = node          # old tail -> new node
        self.tail = node
        self.size += 1

    def to_list(self):
        if self.tail is None:
            return []
        out, cur = [], self.tail.next      # start at head
        while True:
            out.append(cur.data)
            cur = cur.next
            if cur is self.tail.next:      # back at the start
                break
        return out

    def rotate(self):                      # move head to the end in O(1)
        if self.tail:
            self.tail = self.tail.next

players = CircularList()
for p in ["Red", "Green", "Yellow", "Blue"]:
    players.append(p)
turn = players.tail.next
for _ in range(6):                          # six turns of Ludo
    print("Turn:", turn.data)
    turn = turn.next

### Round-Robin Scheduling

An operating system gives each process a fixed **time slice** (quantum). If a process is not finished, it goes to the back of the circle. This is fair: no process waits forever.

In [ ]:
def round_robin(jobs, quantum):
    """jobs: list of (name, burst_time). Returns finishing order and times."""
    ring = CircularList()
    for name, burst in jobs:
        ring.append([name, burst])
    time, finished = 0, []
    prev, cur = ring.tail, ring.tail.next
    while ring.size:
        run = min(quantum, cur.data[1])
        time += run
        cur.data[1] -= run
        if cur.data[1] == 0:                   # job done: remove it from the ring
            finished.append((cur.data[0], time))
            ring.size -= 1
            if ring.size == 0:
                break
            prev.next = cur.next
            if cur is ring.tail:
                ring.tail = prev
            cur = cur.next
        else:
            prev, cur = cur, cur.next
    return finished

print(round_robin([("P1", 5), ("P2", 3), ("P3", 8)], quantum=2))

## 🧩 Part C: Choosing the Right List

In [ ]:
from collections import deque
history = deque(maxlen=3)              # keeps only the last 3 items
for page in ["google.com", "uo.edu.pk", "github.com", "leetcode.com"]:
    history.append(page)
print(history)
history.appendleft("start"); print(history.pop(), history)

## ⚙️ C++ Corner

**C++**

```cpp
#include <iostream>
#include <list>
using namespace std;

struct DNode {
    int data; DNode* prev; DNode* next;
    DNode(int d) : data(d), prev(nullptr), next(nullptr) {}
};

int main() {
    // Manual doubly linked list: 10 <-> 20 <-> 30
    DNode* a = new DNode(10); DNode* b = new DNode(20); DNode* c = new DNode(30);
    a->next = b; b->prev = a; b->next = c; c->prev = b;
    for (DNode* p = c; p; p = p->prev) cout << p->data << " ";   // backwards
    cout << endl;
    // delete b in O(1)
    b->prev->next = b->next; b->next->prev = b->prev; delete b;
    delete a; delete c;

    // The standard library doubly linked list
    list<string> playlist = {"Song A", "Song B", "Song C"};
    playlist.push_front("Intro");
    playlist.pop_back();
    for (auto it = playlist.rbegin(); it != playlist.rend(); ++it) cout << *it << " | ";
}
```

## 🧪 Lab Tasks
Difficulty: 🟢 Easy · 🟡 Medium · 🔴 Hard

### Task 1 🟢 Insert After a Value
Add a method `insert_after(target, data)` to `DoublyLinkedList` (the notebook re-defines the class) that inserts `data` right after the first node containing `target`. Return `True` if inserted, `False` if `target` is not found.

```text
[10, 20, 30].insert_after(20, 25) → [10, 20, 25, 30]
```

In [ ]:
def insert_after(self, target, data):
    # TODO: find the node, then use self._insert_between(...)
    pass

DoublyLinkedList.insert_after = insert_after

In [ ]:
# Test cell: run after your solution
d = DoublyLinkedList()
for x in [10, 20, 30]:
    d.push_back(x)
assert d.insert_after(20, 25) is True and d.forward() == [10, 20, 25, 30]
assert d.insert_after(30, 35) is True and d.backward() == [35, 30, 25, 20, 10]
assert d.insert_after(99, 1) is False and d.size == 5
print("✅ Task 1 passed")

### Task 2 🟢 Reverse a Doubly Linked List
Add `reverse()` to `DoublyLinkedList`: swap `prev` and `next` in **every** node (including the sentinels) and then swap the head and tail sentinels. Both `forward()` and `backward()` must still work.

```text
[1, 2, 3] → [3, 2, 1]
```

In [ ]:
def reverse(self):
    # TODO: swap prev/next in every node, then swap head and tail
    pass

DoublyLinkedList.reverse = reverse

In [ ]:
# Test cell: run after your solution
d = DoublyLinkedList()
for x in [1, 2, 3]:
    d.push_back(x)
d.reverse()
assert d.forward() == [3, 2, 1] and d.backward() == [1, 2, 3]
d.push_back(0); d.push_front(4)
assert d.forward() == [4, 3, 2, 1, 0]
e = DoublyLinkedList(); e.reverse(); assert e.forward() == []
print("✅ Task 2 passed")

### Task 3 🟡 Browser History
Implement `BrowserHistory(homepage)` with methods `visit(url)`, `back(steps)` and `forward(steps)` using a doubly linked list of pages and a `current` reference.

`visit` clears all forward history. `back`/`forward` move at most as far as possible and return the current URL.

```text
b = BrowserHistory("uo.edu.pk"); visit("lms"); visit("results")
back(1) → "lms"; back(5) → "uo.edu.pk"; forward(1) → "lms"
visit("timetable"); forward(2) → "timetable"
```

In [ ]:
class Page:
    def __init__(self, url):
        self.url, self.prev, self.next = url, None, None

class BrowserHistory:
    def __init__(self, homepage):
        self.current = Page(homepage)

    def visit(self, url):
        # TODO: link a new page after current and move to it
        pass

    def back(self, steps):
        # TODO
        pass

    def forward(self, steps):
        # TODO
        pass

In [ ]:
# Test cell: run after your solution
b = BrowserHistory("uo.edu.pk")
b.visit("lms"); b.visit("results")
assert b.back(1) == "lms" and b.back(5) == "uo.edu.pk" and b.forward(1) == "lms"
b.visit("timetable")
assert b.forward(2) == "timetable" and b.back(2) == "uo.edu.pk"
print("✅ Task 3 passed")

### Task 4 🟡 Josephus Problem
n people stand in a circle numbered 1 to n. Starting from person 1, every k-th person is eliminated, and counting continues from the next person. Return the **order of elimination** and the **survivor**.

Simulate it with a circular linked list (O(n·k)). The story: the historian Josephus used this to find the safe position.

```text
josephus(7, 3) → ([3, 6, 2, 7, 5, 1], 4)
```

In [ ]:
def josephus(n, k):
    ring = CircularList()
    for i in range(1, n + 1):
        ring.append(i)
    prev = ring.tail
    cur = ring.tail.next
    order = []
    # TODO: while more than one node remains, move k-1 steps and remove cur
    return order, cur.data

In [ ]:
# Test cell: run after your solution
assert josephus(7, 3) == ([3, 6, 2, 7, 5, 1], 4)
assert josephus(1, 5) == ([], 1)
assert josephus(5, 2) == ([2, 4, 1, 5], 3)
assert josephus(41, 3)[1] == 31
print("✅ Task 4 passed")

### Task 5 🟡 Playlist with Repeat-All
Build a `Playlist` class on a **circular doubly linked list** with: `add(song)`, `next()`, `prev()` (both wrap around), `current()` and `remove_current()` (removes the playing song and moves to the next one).

Return `None` from `current()` when the playlist is empty.

```text
add A, B, C → current A; prev() → C; next() → A; next() → B; remove_current() → now C
```

In [ ]:
class SNode:
    def __init__(self, song):
        self.song, self.prev, self.next = song, None, None

class Playlist:
    def __init__(self):
        self.cur = None
        self.size = 0

    def add(self, song):
        # TODO: first song points to itself; others go before cur (at the end)
        pass

    def current(self):
        return self.cur.song if self.cur else None

    def next(self):
        # TODO
        pass

    def prev(self):
        # TODO
        pass

    def remove_current(self):
        # TODO
        pass

In [ ]:
# Test cell: run after your solution
p = Playlist()
assert p.current() is None
for s in ["A", "B", "C"]:
    p.add(s)
assert p.current() == "A" and p.prev() == "C" and p.next() == "A" and p.next() == "B"
assert p.remove_current() == "B" and p.current() == "C" and p.next() == "A"
p.remove_current(); p.remove_current()
assert p.current() is None and p.size == 0
print("✅ Task 5 passed")

### Task 6 🔴 Delete Duplicates from a Sorted DLL
Write `dedupe(dll)` that removes duplicate values from a **sorted** `DoublyLinkedList` in a single pass using `_unlink`, keeping the first occurrence. Make sure `backward()` still gives the correct result.

```text
[1, 1, 2, 3, 3, 3, 4] → [1, 2, 3, 4]
```

In [ ]:
def dedupe(dll):
    cur = dll.head.next
    # TODO: compare cur with cur.next; unlink duplicates
    return dll

In [ ]:
# Test cell: run after your solution
d = DoublyLinkedList()
for x in [1, 1, 2, 3, 3, 3, 4]:
    d.push_back(x)
dedupe(d)
assert d.forward() == [1, 2, 3, 4] and d.backward() == [4, 3, 2, 1] and d.size == 4
e = DoublyLinkedList(); dedupe(e); assert e.forward() == []
f = DoublyLinkedList()
for x in [5, 5, 5]:
    f.push_back(x)
assert dedupe(f).forward() == [5]
print("✅ Task 6 passed")

### Task 7 🔴 LRU Cache (Interview Classic)
Design an `LRUCache(capacity)` with `get(key)` and `put(key, value)`, **both in O(1)**. When the cache is full, `put` evicts the **least recently used** key. Any `get` or `put` makes a key the most recently used.

Design: a dictionary maps key → DLL node; the DLL keeps keys in order of use (most recent at the front). Use `DNode` with `data = (key, value)`.

```text
c = LRUCache(2); put(1,1); put(2,2); get(1) → 1; put(3,3)  # evicts 2
get(2) → -1; put(4,4)  # evicts 1
get(1) → -1; get(3) → 3; get(4) → 4
```

In [ ]:
class LRUCache:
    def __init__(self, capacity):
        self.capacity = capacity
        self.map = {}                       # key -> node
        self.order = DoublyLinkedList()     # front = most recent

    def get(self, key):
        # TODO: missing -> -1; else move node to the front and return value
        pass

    def put(self, key, value):
        # TODO: update existing key, or evict from the back if full, then add at front
        pass

In [ ]:
# Test cell: run after your solution
c = LRUCache(2)
c.put(1, 1); c.put(2, 2)
assert c.get(1) == 1
c.put(3, 3); assert c.get(2) == -1
c.put(4, 4); assert c.get(1) == -1 and c.get(3) == 3 and c.get(4) == 4
c.put(3, 30); c.put(5, 5); assert c.get(4) == -1 and c.get(3) == 30
print("✅ Task 7 passed")

## 🔬 Bonus: Python’s Own LRU Cache

In [ ]:
# Bonus: run after completing the tasks
from functools import lru_cache
import time

@lru_cache(maxsize=128)               # implemented with a dictionary + doubly linked list
def slow_square(n):
    time.sleep(0.01)                  # pretend this is a slow database or API call
    return n * n

t = time.perf_counter()
for _ in range(3):
    for n in range(20):
        slow_square(n)
print(f"time: {time.perf_counter() - t:.2f}s (only the first round was slow)")
print(slow_square.cache_info())

---
## 🎉 Lab 09 Complete!

Next: **Lab 10: Stacks**

✅ Upload this notebook to your GitHub repository `DSA-Labs/Lab09/`.

---
*Prepared by **Muhammad Ramzan**, MPhil Scholar (AI), PUCIT, University of the Punjab*<br>
🔗 [GitHub](https://github.com/RamzanPUCIT) · [LinkedIn](https://www.linkedin.com/in/ramzan-ai) · [ResearchGate](https://www.researchgate.net/profile/Muhammad-Ramzan-105)